# Solving Sudoku as a Constraint Satisfaction Problem

**Course:** Artificial Intelligence  
**Techniques:** CSP modelling, AC-3 (arc consistency), Backtracking search, MRV heuristic, MAC (maintaining arc consistency)  
**Dataset:** [Kaggle Sudoku (rohanrao)](https://www.kaggle.com/datasets/rohanrao/sudoku) — `puzzle` and `solution` columns, 81-character strings, `0` = empty cell

## Objective
Model Sudoku as a CSP and compare three solving strategies on the same puzzles:

| # | Method | Idea |
|---|--------|------|
| 1 | Plain Backtracking | Fill cells in order, undo on conflict |
| 2 | AC-3 + Backtracking | Prune domains once with AC-3, then plain backtracking |
| 3 | AC-3 + MRV + MAC | Pick the most constrained cell, re-run AC-3 after every assignment |

## Notebook outline
1. Setup and configuration
2. Data loading
3. CSP formulation
4. AC-3 algorithm
5. Search algorithms
6. Demonstration on one puzzle
7. Benchmark (regular puzzles)
8. Benchmark (hard puzzles)
9. Visualisation
10. Discussion

## 1. Setup and Configuration

In [ ]:
import os
import time
import random
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
import matplotlib.pyplot as plt

# ---------------- Configuration ----------------
DATA_PATH   = "/content/drive/MyDrive/sudoku.csv"  # path to sudoku.csv
N_LOAD      = 1000     # puzzles read from the CSV (the file has 1,000,000)
N_REGULAR   = 50       # puzzles used in the regular benchmark (try 100 later)
N_HARD      = 20       # hardest puzzles (most empty cells) used in the hard benchmark
NODE_LIMIT  = 100_000  # safety cap on search nodes per puzzle

Cell    = Tuple[int, int]
Domains = Dict[Cell, Set[int]]

## 2. Data Loading
The notebook mounts Google Drive. If the file is not found, it asks for a manual upload.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("sudoku.csv not found in Drive. Please upload it:")
    files.upload()
    DATA_PATH = "sudoku.csv"

# dtype=str keeps leading zeros of the 81-character puzzle strings
df = pd.read_csv(DATA_PATH, nrows=N_LOAD, dtype=str)
print("Loaded:", df.shape, "| columns:", list(df.columns))
df.head(3)

## 3. CSP Formulation

| CSP component | Sudoku |
|---|---|
| **Variables** | 81 cells, identified by `(row, column)` |
| **Domains** | `{1..9}` for empty cells, `{d}` for a given digit `d` |
| **Constraints** | Every row, column and 3×3 box contains each digit once (all-different) |

Each cell has **20 peers** (8 in its row, 8 in its column, 4 more in its box). A value assigned to a cell must differ from all its peers.

In [ ]:
class SudokuCSP:
    """Sudoku expressed as a CSP: variables, domains and peer constraints."""

    def __init__(self):
        self.cells: List[Cell] = [(r, c) for r in range(9) for c in range(9)]
        self.peers: Dict[Cell, Set[Cell]] = {c: self._peers_of(c) for c in self.cells}

    @staticmethod
    def _peers_of(cell: Cell) -> Set[Cell]:
        r, c = cell
        row = {(r, j) for j in range(9)}
        col = {(i, c) for i in range(9)}
        br, bc = 3 * (r // 3), 3 * (c // 3)
        box = {(br + i, bc + j) for i in range(3) for j in range(3)}
        return (row | col | box) - {cell}

    def initial_domains(self, puzzle: str) -> Domains:
        """Given digit -> singleton domain; empty cell ('0') -> {1..9}."""
        return {
            cell: ({int(ch)} if ch != "0" else set(range(1, 10)))
            for cell, ch in zip(self.cells, puzzle)
        }

    def to_string(self, dom: Domains) -> str:
        """Convert domains to an 81-char string ('0' where a cell is undecided)."""
        return "".join(
            str(next(iter(dom[c]))) if len(dom[c]) == 1 else "0" for c in self.cells
        )

    def is_valid_solution(self, s: str) -> bool:
        """Independent check: complete grid and no peer conflicts."""
        if len(s) != 81 or "0" in s:
            return False
        val = dict(zip(self.cells, s))
        return all(val[c] != val[p] for c in self.cells for p in self.peers[c])


csp = SudokuCSP()
print("Variables (cells):", len(csp.cells))
print("Peers per cell   :", len(csp.peers[(0, 0)]))


def show(s: str) -> None:
    """Print an 81-char string as a 9x9 grid ('_' = empty)."""
    for r in range(9):
        row = s[r * 9:(r + 1) * 9].replace("0", "_")
        print(" ".join(row[:3]), "|", " ".join(row[3:6]), "|", " ".join(row[6:]))
        if r in (2, 5):
            print("------+-------+------")

print("\nPuzzle #0:")
show(df.puzzle[0])
print("\nSolution #0:")
show(df.solution[0])

## 4. Arc Consistency (AC-3)

An arc `(Xi, Xj)` is consistent if every value in `Xi`'s domain has a supporting value in `Xj`'s domain.  
For Sudoku's "not equal" constraint, a value `v` in `Xi` loses support only when `Xj`'s domain is exactly `{v}`.

**Procedure:** put all arcs in a queue; `revise` removes unsupported values; when a domain shrinks, re-queue the arcs pointing at that cell. An empty domain means the (partial) assignment is inconsistent.

In [ ]:
def revise(dom: Domains, xi: Cell, xj: Cell) -> bool:
    """Remove from dom[xi] any value that has no support in dom[xj]."""
    if len(dom[xj]) == 1:
        v = next(iter(dom[xj]))
        if v in dom[xi]:
            dom[xi].discard(v)
            return True
    return False


def ac3(csp: SudokuCSP, dom: Domains, arcs: Optional[List[Tuple[Cell, Cell]]] = None) -> bool:
    """
    Enforce arc consistency in place.
    arcs=None -> start with every arc; otherwise start with the given arcs only.
    Returns False if some domain becomes empty (inconsistent), else True.
    """
    if arcs is None:
        arcs = [(xi, xj) for xi in csp.cells for xj in csp.peers[xi]]
    queue = deque(arcs)
    while queue:
        xi, xj = queue.popleft()
        if revise(dom, xi, xj):
            if not dom[xi]:
                return False
            for xk in csp.peers[xi]:
                if xk != xj:
                    queue.append((xk, xi))
    return True

## 5. Search Algorithms

All three solvers return a `Result` with the same fields so they can be compared fairly.

* **Node** = one value tried for a cell.  
* **Backtrack** = a tried value that led to a dead end and was undone.

In [ ]:
@dataclass
class Result:
    solution: Optional[str]
    backtracks: int
    nodes: int
    time_s: float


class NodeLimitExceeded(Exception):
    pass


class _Counter:
    def __init__(self, limit: int):
        self.nodes = 0
        self.backtracks = 0
        self.limit = limit

    def node(self):
        self.nodes += 1
        if self.nodes > self.limit:
            raise NodeLimitExceeded


# ---------- Method 1: plain backtracking (no inference) ----------
def solve_plain(csp: SudokuCSP, puzzle: str, node_limit: int = NODE_LIMIT) -> Result:
    t0 = time.perf_counter()
    idx = {c: i for i, c in enumerate(csp.cells)}
    peer_idx = [[idx[p] for p in csp.peers[c]] for c in csp.cells]
    grid = [int(ch) for ch in puzzle]
    cnt = _Counter(node_limit)

    def bt(i: int) -> bool:
        while i < 81 and grid[i] != 0:
            i += 1
        if i == 81:
            return True
        for v in range(1, 10):
            if all(grid[p] != v for p in peer_idx[i]):
                cnt.node()
                grid[i] = v
                if bt(i + 1):
                    return True
                grid[i] = 0
                cnt.backtracks += 1
        return False

    try:
        ok = bt(0)
    except NodeLimitExceeded:
        ok = False
    sol = "".join(map(str, grid)) if ok else None
    return Result(sol, cnt.backtracks, cnt.nodes, time.perf_counter() - t0)


# ---------- Method 2: AC-3 once, then backtracking in fixed order ----------
def solve_ac3_bt(csp: SudokuCSP, puzzle: str, node_limit: int = NODE_LIMIT) -> Result:
    t0 = time.perf_counter()
    cnt = _Counter(node_limit)
    dom = csp.initial_domains(puzzle)
    if not ac3(csp, dom):
        return Result(None, 0, 0, time.perf_counter() - t0)

    def bt(dom: Domains) -> Optional[Domains]:
        free = [c for c in csp.cells if len(dom[c]) > 1]
        if not free:
            return dom
        x = free[0]                                   # fixed order (no MRV)
        for v in sorted(dom[x]):
            if any(len(dom[p]) == 1 and v in dom[p] for p in csp.peers[x]):
                continue                              # conflicts with a decided peer
            cnt.node()
            new = {k: set(d) for k, d in dom.items()}
            new[x] = {v}                              # no propagation (no MAC)
            res = bt(new)
            if res is not None:
                return res
            cnt.backtracks += 1
        return None

    try:
        res = bt(dom)
    except NodeLimitExceeded:
        res = None
    sol = csp.to_string(res) if res is not None else None
    return Result(sol, cnt.backtracks, cnt.nodes, time.perf_counter() - t0)


# ---------- Method 3: AC-3 + MRV + MAC ----------
def solve_ac3_mrv_mac(csp: SudokuCSP, puzzle: str, node_limit: int = NODE_LIMIT) -> Result:
    t0 = time.perf_counter()
    cnt = _Counter(node_limit)
    dom = csp.initial_domains(puzzle)
    if not ac3(csp, dom):
        return Result(None, 0, 0, time.perf_counter() - t0)

    def bt(dom: Domains) -> Optional[Domains]:
        free = [c for c in csp.cells if len(dom[c]) > 1]
        if not free:
            return dom
        x = min(free, key=lambda c: len(dom[c]))      # MRV: smallest domain first
        for v in sorted(dom[x]):
            cnt.node()
            new = {k: set(d) for k, d in dom.items()}
            new[x] = {v}
            arcs = [(p, x) for p in csp.peers[x]]     # MAC: propagate from x only
            if ac3(csp, new, arcs):
                res = bt(new)
                if res is not None:
                    return res
            cnt.backtracks += 1
        return None

    try:
        res = bt(dom)
    except NodeLimitExceeded:
        res = None
    sol = csp.to_string(res) if res is not None else None
    return Result(sol, cnt.backtracks, cnt.nodes, time.perf_counter() - t0)


SOLVERS = {
    "Plain Backtracking":  solve_plain,
    "AC-3 + Backtracking": solve_ac3_bt,
    "AC-3 + MRV + MAC":    solve_ac3_mrv_mac,
}
print("Solvers ready:", list(SOLVERS))

## 6. Demonstration on a Single Puzzle

In [ ]:
puzzle, truth = df.puzzle[0], df.solution[0]

# (a) What does AC-3 alone achieve?
dom = csp.initial_domains(puzzle)
given = sum(len(d) == 1 for d in dom.values())
ac3(csp, dom)
decided = sum(len(d) == 1 for d in dom.values())
print(f"Cells decided: {given} given  ->  {decided} after AC-3 alone (of 81)")
print("AC-3 alone solved it:", csp.to_string(dom) == truth)

# (b) Each full solver on the same puzzle
print()
for name, solver in SOLVERS.items():
    r = solver(csp, puzzle)
    print(f"{name:22s} correct={r.solution == truth!s:5s} "
          f"valid={csp.is_valid_solution(r.solution or '')!s:5s} "
          f"backtracks={r.backtracks:5d} nodes={r.nodes:5d} time={r.time_s*1000:7.2f} ms")

## 7. Benchmark on Regular Puzzles
Every solver runs on exactly the same puzzles. A puzzle counts as **solved** when the returned grid equals the dataset's `solution`.

In [ ]:
def benchmark(data: pd.DataFrame) -> pd.DataFrame:
    """Run every solver on every puzzle; return one row per (puzzle, method)."""
    rows = []
    for i, (q, truth) in enumerate(zip(data.puzzle, data.solution)):
        for name, solver in SOLVERS.items():
            r = solver(csp, q)
            rows.append({
                "puzzle": i,
                "method": name,
                "solved": r.solution == truth,
                "backtracks": r.backtracks,
                "nodes": r.nodes,
                "time_s": r.time_s,
            })
    return pd.DataFrame(rows)


def summarise(res: pd.DataFrame) -> pd.DataFrame:
    n = res.puzzle.nunique()
    out = res.groupby("method", sort=False).agg(
        solved=("solved", "sum"),
        avg_backtracks=("backtracks", "mean"),
        avg_nodes=("nodes", "mean"),
        max_backtracks=("backtracks", "max"),
        avg_time_ms=("time_s", lambda s: s.mean() * 1000),
        total_time_s=("time_s", "sum"),
    )
    out["solved"] = out["solved"].astype(int).astype(str) + f"/{n}"
    return out.round(2)


regular = df.head(N_REGULAR)
res_regular = benchmark(regular)
summary_regular = summarise(res_regular)
summary_regular

## 8. Benchmark on Hard Puzzles
Puzzles with the **most empty cells** are used as a proxy for difficulty.

In [ ]:
df["blanks"] = df.puzzle.str.count("0")
hard = df.sort_values("blanks", ascending=False).head(N_HARD)
print("Empty cells in the selected hard puzzles:", hard.blanks.min(), "to", hard.blanks.max())

res_hard = benchmark(hard)
summary_hard = summarise(res_hard)
summary_hard

## 9. Visualisation

In [ ]:
def plot_summary(summary: pd.DataFrame, title: str) -> None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    names = list(summary.index)
    colors = ["#9aa5b1", "#4f8fc0", "#2e9e6e"]

    b1 = ax[0].bar(names, summary["avg_backtracks"], color=colors)
    ax[0].set_title("Average backtracks per puzzle")
    ax[0].set_yscale("symlog")
    ax[0].bar_label(b1, fmt="%.1f")

    b2 = ax[1].bar(names, summary["avg_time_ms"], color=colors)
    ax[1].set_title("Average time per puzzle (ms)")
    ax[1].bar_label(b2, fmt="%.1f")

    for a in ax:
        a.tick_params(axis="x", rotation=15)
        a.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_summary(summary_regular, f"Regular puzzles (n={N_REGULAR})")
plot_summary(summary_hard,    f"Hard puzzles (n={N_HARD})")

## 10. Discussion
*Fill this section using the tables and charts produced above.*

**What to compare**
* **Search effort** — average and maximum backtracks per method (hardware-independent).
* **Runtime** — average time per puzzle. AC-3 adds work at every node, so on easy puzzles its overhead can outweigh the savings; on hard puzzles the saved search usually matters more.
* **Reliability** — how many puzzles each method solved within the node limit.

**Why the methods behave differently**
* *Plain backtracking* only detects a conflict when it is about to place a clashing value, so it explores many dead ends.
* *AC-3* removes impossible values before any guessing, shrinking the search space; many easy puzzles are solved by AC-3 alone (Section 6).
* *MRV* branches on the cell with the fewest options, so wrong guesses fail quickly.
* *MAC* re-runs propagation after every guess, so a wrong guess is usually detected immediately instead of several levels deeper.

**Conclusion**
Summarise in 2–3 sentences which combination performed best on your data, and support it with the numbers from Sections 7 and 8.